## Imports

In [1]:

import tensorflow as tf
import os
import sys


from preprocess import create_data_generators, get_class_map

# import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

from pathlib import Path

## Constants


In [2]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

## Paths

In [3]:
from pathlib import Path
import os

PROJECT_BASE_DIR = Path(os.environ["NEURODETECT_BASE_DIR"])

TRAIN_DIR = str(PROJECT_BASE_DIR / "train")
TEST_DIR = str(PROJECT_BASE_DIR / "test")
MODELS_DIR = PROJECT_BASE_DIR / "models"

print("Project base:", PROJECT_BASE_DIR)
print("Train path:", TRAIN_DIR)
print("Test path:", TEST_DIR)
print("Models path:", MODELS_DIR)

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Test exists:", os.path.exists(TEST_DIR))
print("Models exists:", os.path.exists(MODELS_DIR))

Project base: G:\My Drive\SeniorProject\datasets\dataset
Train path: G:\My Drive\SeniorProject\datasets\dataset\train
Test path: G:\My Drive\SeniorProject\datasets\dataset\test
Models path: G:\My Drive\SeniorProject\datasets\dataset\models
Train exists: True
Test exists: True
Models exists: True


## Train Gen

In [4]:
train_generator, test_generator = create_data_generators(
    TRAIN_DIR,
    TEST_DIR,
    IMG_SIZE,
    BATCH_SIZE
)

labels = get_class_map(train_generator)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


## Batch

In [5]:
X_batch, y_batch = next(train_generator) 
print("Batch image shape:", X_batch.shape)  

print("\nLoading Test Data:")

X_batch, y_batch = next(test_generator)  
print("Batch image shape:", X_batch.shape) 

Batch image shape: (32, 224, 224, 3)

Loading Test Data:
Batch image shape: (32, 224, 224, 3)


## Baseline Model

In [6]:
num_classes = len(labels)

model_baseline = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model_baseline.summary()

d:\SE\SP\SeniorProject\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

## Compile

In [7]:
model_baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Run Epochs

In [8]:
import time
import math

steps_per_epoch = math.ceil(train_generator.samples / BATCH_SIZE)
validation_steps = math.ceil(test_generator.samples / BATCH_SIZE)

start = time.time()
history_baseline = model_baseline.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=test_generator,
    validation_steps=validation_steps,
    epochs=EPOCHS
)
train_time_sec = time.time() - start

print("Training time (sec):", round(train_time_sec, 2))

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 2464s 27s/step - accuracy: 0.4282 - loss: 1.2306 - val_accuracy: 0.3020 - val_loss: 2.2144
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 114s 1s/step - accuracy: 0.5495 - loss: 1.0239 - val_accuracy: 0.3376 - val_loss: 1.9677
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 103s 1s/step - accuracy: 0.5930 - loss: 0.9461 - val_accuracy: 0.3223 - val_loss: 2.7411
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.6296 - loss: 0.8763 - val_accuracy: 0.3832 - val_loss: 3.0832
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 102s 1s/step - accuracy: 0.6449 - loss: 0.8500 - val_accuracy: 0.3807 - val_loss: 2.3284
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 100s 1s/step - accuracy: 0.6693 - loss: 0.7794 - val_accuracy: 0.4289 - val_loss: 2.4565
Epoch 7/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 104s 1s/step - accuracy: 0.6875 - loss: 0.7382 - val_accuracy: 0.3934 - val_loss: 3.6345
Epoch 8/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 107s 1s/step - accuracy: 0.6808 - loss: 0.7379 - val_accuracy: 0.3680 -

## Results

In [9]:
baseline_loss, baseline_acc = model_baseline.evaluate(
    test_generator,
    steps=validation_steps
)
print("Baseline Loss:", baseline_loss)
print("Baseline Accuracy:", baseline_acc)

13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 977ms/step - accuracy: 0.4365 - loss: 2.6386
Baseline Loss: 2.638627767562866
Baseline Accuracy: 0.43654823303222656


## Analysis

In [10]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# reset generator to start
test_generator.reset()

y_prob = model_baseline.predict(test_generator, steps=validation_steps)
y_pred = np.argmax(y_prob, axis=1)

y_true = test_generator.classes  # integer labels in directory order
class_names = list(test_generator.class_indices.keys())

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

13/13 ━━━━━━━━━━━━━━━━━━━━ 13s 977ms/step
Confusion Matrix:
 [[  7  28  62   3]
 [  0  35  75   5]
 [  0   2 103   0]
 [  2   8  37  27]]

Classification Report:

                  precision    recall  f1-score   support

    glioma_tumor       0.78      0.07      0.13       100
meningioma_tumor       0.48      0.30      0.37       115
        no_tumor       0.37      0.98      0.54       105
 pituitary_tumor       0.77      0.36      0.50        74

        accuracy                           0.44       394
       macro avg       0.60      0.43      0.38       394
    weighted avg       0.58      0.44      0.38       394



## Save Model

In [11]:
os.makedirs(MODELS_DIR, exist_ok=True)

baseline_model_path = MODELS_DIR / "baseline_cnn.keras"
model_baseline.save(str(baseline_model_path))
print(f"Saved to {baseline_model_path}")

Saved to G:\My Drive\SeniorProject\datasets\dataset\models\baseline_cnn.keras


# Transfer Model

In [12]:
transfer_train_datagen= ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
)
transfer_test_datagen=ImageDataGenerator(
    preprocessing_function=preprocess_input
)
train_gen_tf=transfer_train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
test_gen_tf=transfer_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

Found 2870 images belonging to 4 classes.
Found 394 images belonging to 4 classes.


In [13]:
base_model=MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base_model.trainable = False

X= base_model.output
X=GlobalAveragePooling2D()(X)
X=Dense(128, activation='relu')(X)
X=Dropout(0.5)(X)
outputs= Dense(train_gen_tf.num_classes, activation='softmax')(X)
model_transfer=Model(inputs=base_model.input, outputs=outputs)

## Compile

In [14]:
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Epochs

In [15]:
EPOCHS_TL = 20
history_transfer= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=EPOCHS_TL
)

Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 225s 2s/step - accuracy: 0.6780 - loss: 0.7898 - val_accuracy: 0.5761 - val_loss: 1.1035
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 208s 2s/step - accuracy: 0.7889 - loss: 0.5292 - val_accuracy: 0.5812 - val_loss: 1.1283
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 201s 2s/step - accuracy: 0.8118 - loss: 0.4605 - val_accuracy: 0.5863 - val_loss: 1.1013
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 171s 2s/step - accuracy: 0.8418 - loss: 0.4232 - val_accuracy: 0.6244 - val_loss: 1.0972
Epoch 5/20
67/90 ━━━━━━━━━━━━━━━━━━━━ 1:26 4s/step - accuracy: 0.8576 - loss: 0.3761

UnknownError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
OSError: [Errno 28] No space left on device
Traceback (most recent call last):

  File "d:\SE\SP\SeniorProject\venv\Lib\site-packages\tensorflow\python\ops\script_ops.py", line 269, in __call__
    ret = func(*args)

  File "d:\SE\SP\SeniorProject\venv\Lib\site-packages\tensorflow\python\autograph\impl\api.py", line 643, in wrapper
    return func(*args, **kwargs)

  File "d:\SE\SP\SeniorProject\venv\Lib\site-packages\tensorflow\python\data\ops\from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "d:\SE\SP\SeniorProject\venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py", line 264, in _finite_generator
    yield self._standardize_batch(self.py_dataset[i])
                                  ~~~~~~~~~~~~~~~^^^

  File "d:\SE\SP\SeniorProject\venv\Lib\site-packages\keras\

## Results

In [ ]:
transfer_loss, transfer_acc = model_transfer.evaluate(test_gen_tf)
print("Transfer Learning Loss:", transfer_loss)

13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 934ms/step - accuracy: 0.7386 - loss: 1.5323
Transfer Learning Loss: 1.5322575569152832


## Comparision

In [ ]:
print('Baseline Accuracy:', baseline_acc)
print('Transfer Learning Accuracy:', transfer_acc)

NameError: name 'baseline_acc' is not defined

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
model_transfer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
historical_finetune= model_transfer.fit(
    train_gen_tf,
    validation_data=test_gen_tf,
    epochs=10
)

## Save Model

In [ ]:
os.makedirs(MODELS_DIR, exist_ok=True)

transfer_model_path = MODELS_DIR / "mobilenetv2_transfer.keras"
model_transfer.save(str(transfer_model_path))
print(f"Saved to {transfer_model_path}")